In [1]:
pip install pulp

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

food_candidates = pd.read_csv("./ready_food_data_frame.csv")


In [3]:
patient_target = {
    "diet_type": "DM_HT_OBESITY",
    "energy_kcal": 1600,
    "carb_g": 240,
    "protein_g": 60,
    "fat_g": 44,
    "sodium_mg_max": 2000,
    "fiber_g_min": 25
}

In [4]:
def get_disease_constraints(diet_type, energy_kcal):
    """
    diet_type options:
    HT
    HT_OBESITY
    DM
    DM_OBESITY
    DM_HT
    DM_HT_OBESITY
    """

    constraints = {
        "energy_target": energy_kcal,
        "energy_mode": "maintenance",
        "carb_min_g": None,
        "carb_max_g": None,
        "fat_min_g": None,
        "fat_max_g": None,
        "sodium_max_mg": None,
        "potassium_min_mg": None,
        "calcium_min_mg": None,
        "fiber_min_g": None,
        "vegetable_fruit_min_portions": None,
        "sugar_max_portions": None
    }

    # 1. Hipertensi tanpa obesitas
    if diet_type == "HT":
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 2. Hipertensi dengan obesitas
    elif diet_type == "HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = energy_kcal * 0.55 / 4
        constraints["carb_max_g"] = energy_kcal * 0.65 / 4
        constraints["fat_min_g"] = energy_kcal * 0.20 / 9
        constraints["fat_max_g"] = energy_kcal * 0.25 / 9
        constraints["sodium_max_mg"] = 2300
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 3. DM tanpa obesitas
    elif diet_type == "DM":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sugar_max_portions"] = 0

    # 4. DM dengan obesitas
    elif diet_type == "DM_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 5. DM + Hipertensi tanpa obesitas
    elif diet_type == "DM_HT":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2300
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 6. DM + Hipertensi + Obesitas
    elif diet_type == "DM_HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = 130
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2000
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    return constraints

In [5]:
portion_plan = {
    1100: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 1, "SS": 0, "M": 3, "G": 1},
    1200: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 1, "SS": 1, "M": 3, "G": 2},
    1300: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 2, "SS": 1, "M": 4, "G": 2},
    1400: {"MP": 3, "LH": 3, "LN": 3, "S": 2, "B": 2, "SS": 0, "M": 3, "G": 3},
    1500: {"MP": 3, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 0, "M": 4, "G": 3},
    1600: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 2, "SS": 0, "M": 4, "G": 2},
    1700: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 3, "G": 2},
    1800: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 4, "G": 3},
    1900: {"MP": 4, "LH": 4, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 4, "G": 3},
    2000: {"MP": 4, "LH": 3, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 4, "G": 4},
    2100: {"MP": 4, "LH": 3, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 4, "G": 4},
    2200: {"MP": 4, "LH": 3, "LN": 3, "S": 4, "B": 4, "SS": 2, "M": 5, "G": 4},
    2300: {"MP": 5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 0, "M": 5, "G": 4},
    2400: {"MP": 5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 5, "G": 4},
    2500: {"MP": 5.5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 5, "G": 4},
}

In [6]:
# ============================================================
# 3. Pick nearest available energy level
# ============================================================

def get_nearest_energy_level(energy_kcal, available_levels):
    return min(available_levels, key=lambda x: abs(x - energy_kcal))


def adjust_portion_plan_for_disease(base_plan, diet_type):
    plan = base_plan.copy()

    # DM: sugar should be avoided or minimized
    if "DM" in diet_type:
        plan["G"] = 0

    # Obesity: reduce sugar and oil portions
    if "OBESITY" in diet_type:
        plan["G"] = 0
        if "M" in plan:
            plan["M"] = max(0, plan["M"] - 1)

    # DM obesity / DM+HT: ensure vegetable + fruit >= 5 portions
    if diet_type in ["DM_OBESITY", "DM_HT", "DM_HT_OBESITY"]:
        current_sb = plan.get("S", 0) + plan.get("B", 0)
        if current_sb < 5:
            plan["S"] = plan.get("S", 0) + (5 - current_sb)

    return plan

In [7]:
def filter_foods_for_disease(df, diet_type):
    filtered = df.copy()

    # DM: remove sugar category if exists
    if "DM" in diet_type:
        filtered = filtered[filtered["category_code"] != "G"]

    # Hypertension: remove very high sodium foods
    if "HT" in diet_type:
        filtered = filtered[
            filtered["sodium_mg_per_portion"].isna() |
            (filtered["sodium_mg_per_portion"] <= 400)
        ]

    # Obesity: remove very high energy per portion
    if "OBESITY" in diet_type:
        filtered = filtered[
            (
                filtered["category_code"].isin(["MP", "LN", "LH"])
            )
            |
            (
                filtered["energy_kcal_per_portion"] <= 250
            )
        ]

    return filtered


candidate_for_patient = filter_foods_for_disease(
    food_candidates,
    patient_target["diet_type"]
)

print("Candidate foods after disease filter:", len(candidate_for_patient))

display(
    candidate_for_patient[[
        "food_code",
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
        "energy_kcal_per_portion",
        "carb_g_per_portion",
        "sodium_mg_per_portion",
        "fiber_g_per_portion"
    ]].head(20)
)

Candidate foods after disease filter: 98


,food_code,food_name,category_code,urt,gram_per_portion,energy_kcal_per_portion,carb_g_per_portion,sodium_mg_per_portion,fiber_g_per_portion
0,AR001,"Beras giling, mentah",MP,3/4 Gelas,100.0,357.00,77.100,27.00,0.200
1,AR002,"Beras giling var pelita, mentah",MP,3/4 Gelas,100.0,369.00,77.100,0.00,0.400
2,AR003,"Beras giling var rojolele, mentah",MP,3/4 Gelas,100.0,357.00,77.100,112.90,0.200
3,AR004,"Beras hitam, mentah",MP,3/4 Gelas,100.0,351.00,76.900,15.00,20.100
4,AR013,"Beras tumbuk merah, mentah",MP,3/4 Gelas,100.0,352.00,76.200,10.00,0.800
5,AP006,"Bihun, mentah",MP,1/2 Gelas,50.0,174.00,41.050,2.50,0.600
6,AP020,"Makaroni, mentah",MP,1½ Gelas,50.0,176.50,39.350,2.50,2.450
7,AP024,Roti putih,MP,3 Iris,70.0,173.60,35.000,63.70,6.370
8,AP029,Biskuit,MP,4 Buah Besar,40.0,183.20,30.040,8.12,0.840
9,BR013,"Kentang, segar",MP,2 Buah Sedang,210.0,130.20,28.350,14.70,1.050


In [8]:
# ============================================================
# 1. Patient input
# ============================================================


diet_type = patient_target["diet_type"]
energy_target = patient_target["energy_kcal"]


# ============================================================
# 2. Create disease constraints
# ============================================================

constraints = get_disease_constraints(
    diet_type=diet_type,
    energy_kcal=energy_target
)


# ============================================================
# 3. Get portion plan
# ============================================================

nearest_energy = get_nearest_energy_level(
    energy_target,
    list(portion_plan.keys())
)

base_plan = portion_plan[nearest_energy]

adjusted_plan = adjust_portion_plan_for_disease(
    base_plan,
    diet_type
)


# ============================================================
# 4. Filter food candidates
# ============================================================

candidate_for_patient = filter_foods_for_disease(
    food_candidates,
    diet_type
)

milp_df = candidate_for_patient.copy().reset_index(drop=True)


# ============================================================
# 5. Clean numeric columns
# ============================================================

numeric_cols = [
    "gram_per_portion",
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

for col in numeric_cols:
    if col in milp_df.columns:
        milp_df[col] = pd.to_numeric(milp_df[col], errors="coerce").fillna(0)


# ============================================================
# 6. Check result
# ============================================================

print("Diet type:", diet_type)
print("Energy target:", energy_target)
print("Nearest energy level:", nearest_energy)
print("Constraints:", constraints)
print("Adjusted portion plan:", adjusted_plan)
print("Total food candidates:", len(milp_df))

Diet type: DM_HT_OBESITY
Energy target: 1600
Nearest energy level: 1600
Constraints: {'energy_target': 1600, 'energy_mode': 'deficit', 'carb_min_g': 130, 'carb_max_g': None, 'fat_min_g': None, 'fat_max_g': None, 'sodium_max_mg': 2000, 'potassium_min_mg': None, 'calcium_min_mg': None, 'fiber_min_g': 40.0, 'vegetable_fruit_min_portions': 5, 'sugar_max_portions': 0}
Adjusted portion plan: {'MP': 4, 'LH': 3, 'LN': 3, 'S': 3, 'B': 2, 'SS': 0, 'M': 3, 'G': 0}
Total food candidates: 98


In [9]:
import pandas as pd
import numpy as np
import pulp

# ============================================================
# 1. Prepare MILP input data
# ============================================================

milp_df = candidate_for_patient.copy().reset_index(drop=True)

# Required nutrient columns
required_cols = [
    "food_name",
    "category_code",
    "urt",
    "gram_per_portion",
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "fiber_g_per_portion"
]

# Keep only rows with required values
milp_df = milp_df.dropna(subset=[
    "category_code",
    "energy_kcal_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "fiber_g_per_portion"
]).copy()

# Make sure numeric columns are numeric
numeric_cols = [
    "gram_per_portion",
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

for col in numeric_cols:
    if col in milp_df.columns:
        milp_df[col] = pd.to_numeric(milp_df[col], errors="coerce").fillna(0)

print("Total MILP candidate foods:", len(milp_df))

display(
    milp_df.groupby("category_code")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

Total MILP candidate foods: 91


,category_code,count
5,S,40
0,B,17
4,MP,11
1,LH,9
2,LN,7
3,M,5
6,SS,2


In [10]:
def reduce_candidates_for_speed(df):
    reduced_parts = []

    category_limits = {
        "MP": 10,
        "LH": 10,
        "LN": 10,
        "S": 30,
        "B": 15,
        "M": 5,
        "SS": 5
    }

    for category, limit in category_limits.items():
        temp = df[df["category_code"] == category].copy()

        if category == "S":
            temp = temp.sort_values(
                by=["fiber_g_per_portion", "sodium_mg_per_portion"],
                ascending=[False, True]
            ).head(limit)

        elif category == "B":
            temp = temp.sort_values(
                by=["fiber_g_per_portion", "energy_kcal_per_portion"],
                ascending=[False, True]
            ).head(limit)

        else:
            temp = temp.sort_values(
                by=["sodium_mg_per_portion", "energy_kcal_per_portion"],
                ascending=[True, True]
            ).head(limit)

        reduced_parts.append(temp)

    return pd.concat(reduced_parts).reset_index(drop=True)


milp_df = reduce_candidates_for_speed(milp_df)

print("Reduced MILP candidate foods:", len(milp_df))

Reduced MILP candidate foods: 78


In [11]:
# ============================================================
# 1. Prepare 7-day MILP model - optimized version
# ============================================================

days = list(range(1, 8))

portion_step = 0.5
max_portion_per_food = 2.0
max_units = int(max_portion_per_food / portion_step)

model = pulp.LpProblem(
    "Seven_Day_DM_HT_Obesity_Menu_Recommendation",
    pulp.LpMinimize
)

# x[d, i] = integer portion unit
# 0 = not selected
# 1 = 0.5 portion
# 2 = 1 portion
# 3 = 1.5 portions
# 4 = 2 portions
x = {
    (d, i): pulp.LpVariable(
        f"x_day{d}_food{i}",
        lowBound=0,
        upBound=max_units,
        cat="Integer"
    )
    for d in days
    for i in milp_df.index
}

# Real portion used in calculation
portion = {
    (d, i): x[(d, i)] * portion_step
    for d in days
    for i in milp_df.index
}

In [12]:
def get_disease_constraints(diet_type, energy_kcal):
    """
    diet_type options:
    HT
    HT_OBESITY
    DM
    DM_OBESITY
    DM_HT
    DM_HT_OBESITY
    """

    constraints = {
        "energy_target": energy_kcal,
        "energy_mode": "maintenance",
        "carb_min_g": None,
        "carb_max_g": None,
        "fat_min_g": None,
        "fat_max_g": None,
        "sodium_max_mg": None,
        "potassium_min_mg": None,
        "calcium_min_mg": None,
        "fiber_min_g": None,
        "vegetable_fruit_min_portions": None,
        "sugar_max_portions": None
    }

    # 1. Hipertensi tanpa obesitas
    if diet_type == "HT":
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 2. Hipertensi dengan obesitas
    elif diet_type == "HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = energy_kcal * 0.55 / 4
        constraints["carb_max_g"] = energy_kcal * 0.65 / 4
        constraints["fat_min_g"] = energy_kcal * 0.20 / 9
        constraints["fat_max_g"] = energy_kcal * 0.25 / 9
        constraints["sodium_max_mg"] = 2300
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 3. DM tanpa obesitas
    elif diet_type == "DM":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sugar_max_portions"] = 0

    # 4. DM dengan obesitas
    elif diet_type == "DM_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 5. DM + Hipertensi tanpa obesitas
    elif diet_type == "DM_HT":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2300
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 6. DM + Hipertensi + Obesitas
    elif diet_type == "DM_HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = 130
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2000
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    return constraints

In [13]:
# ============================================================
# 2. Daily nutrient totals and constraints
# ============================================================

daily_energy = {}
daily_protein = {}
daily_fat = {}
daily_carb = {}
daily_sodium = {}
daily_fiber = {}

energy_target = constraints["energy_target"]
energy_min = energy_target * 0.90
energy_max = energy_target * 1.10

for d in days:

    daily_energy[d] = pulp.lpSum(
        portion[(d, i)] * milp_df.loc[i, "energy_kcal_per_portion"]
        for i in milp_df.index
    )

    daily_protein[d] = pulp.lpSum(
        portion[(d, i)] * milp_df.loc[i, "protein_g_per_portion"]
        for i in milp_df.index
    )

    daily_fat[d] = pulp.lpSum(
        portion[(d, i)] * milp_df.loc[i, "fat_g_per_portion"]
        for i in milp_df.index
    )

    daily_carb[d] = pulp.lpSum(
        portion[(d, i)] * milp_df.loc[i, "carb_g_per_portion"]
        for i in milp_df.index
    )

    daily_sodium[d] = pulp.lpSum(
        portion[(d, i)] * milp_df.loc[i, "sodium_mg_per_portion"]
        for i in milp_df.index
    )

    daily_fiber[d] = pulp.lpSum(
        portion[(d, i)] * milp_df.loc[i, "fiber_g_per_portion"]
        for i in milp_df.index
    )

    # Daily energy target must pass
    model += daily_energy[d] >= energy_min, f"energy_min_day{d}"
    model += daily_energy[d] <= energy_max, f"energy_max_day{d}"

    # Daily carb minimum
    if constraints.get("carb_min_g") is not None:
        model += daily_carb[d] >= constraints["carb_min_g"], f"carb_min_day{d}"

    # Daily carb maximum, if available
    if constraints.get("carb_max_g") is not None:
        model += daily_carb[d] <= constraints["carb_max_g"], f"carb_max_day{d}"

    # Daily fat minimum, if available
    if constraints.get("fat_min_g") is not None:
        model += daily_fat[d] >= constraints["fat_min_g"], f"fat_min_day{d}"

    # Daily fat maximum, if available
    if constraints.get("fat_max_g") is not None:
        model += daily_fat[d] <= constraints["fat_max_g"], f"fat_max_day{d}"

    # Daily sodium maximum
    if constraints.get("sodium_max_mg") is not None:
        model += daily_sodium[d] <= constraints["sodium_max_mg"], f"sodium_max_day{d}"

    # Daily fiber minimum
    if constraints.get("fiber_min_g") is not None:
        model += daily_fiber[d] >= constraints["fiber_min_g"], f"fiber_min_day{d}"

In [14]:
# ============================================================
# 3. Daily category portion constraints
# ============================================================

for d in days:
    for category, required_portion in adjusted_plan.items():

        category_indices = milp_df[
            milp_df["category_code"] == category
        ].index.tolist()

        if len(category_indices) == 0 and required_portion == 0:
            continue

        if len(category_indices) == 0 and required_portion > 0:
            print(
                f"Warning: no candidate food for category {category}, "
                f"but required portion is {required_portion}"
            )
            continue

        model += (
            pulp.lpSum(portion[(d, i)] for i in category_indices) == required_portion,
            f"portion_day{d}_{category}"
        )

In [15]:
def make_variety_key(row):
    name = str(row["food_name"]).lower()
    category = str(row["category_code"])

    # LH - animal protein
    if category == "LH":
        if "cumi" in name:
            return "LH_cumi"
        if "sapi" in name or "daging sapi" in name:
            return "LH_daging_sapi"
        if "ayam" in name:
            return "LH_ayam"
        if "lele" in name:
            return "LH_ikan_lele"
        if "belut" in name:
            return "LH_belut"
        if "ikan mas" in name:
            return "LH_ikan_mas"

    # LN - plant protein
    if category == "LN":
        if "tempe" in name:
            return "LN_tempe"
        if "tahu" in name:
            return "LN_tahu"
        if "kacang kedelai" in name:
            return "LN_kacang_kedelai"
        if "kacang merah" in name:
            return "LN_kacang_merah"
        if "kacang hijau" in name:
            return "LN_kacang_hijau"

    # MP - staple food
    if category == "MP":
        if "makaroni" in name:
            return "MP_makaroni"
        if "bihun" in name:
            return "MP_bihun"
        if "nasi" in name or "beras" in name:
            return "MP_nasi"
        if "roti" in name:
            return "MP_roti"
        if "biskuit" in name:
            return "MP_biskuit"
        if "kentang" in name:
            return "MP_kentang"

    # B - fruit
    if category == "B":
        if "duku" in name:
            return "B_duku"
        if "pepaya" in name:
            return "B_pepaya"
        if "apel" in name:
            return "B_apel"
        if "pisang" in name:
            return "B_pisang"

    # S - vegetable
    if category == "S":
        if "daun melinjo" in name:
            return "S_daun_melinjo"
        if "nangka muda" in name:
            return "S_nangka_muda"
        if "kangkung" in name:
            return "S_kangkung"
        if "bayam" in name:
            return "S_bayam"
        if "rebung" in name:
            return "S_rebung"
        if "komak" in name:
            return "S_komak"

    # M - oil/fat
    if category == "M":
        if "minyak kacang" in name:
            return "M_minyak_kacang"
        if "minyak kelapa" in name:
            return "M_minyak_kelapa"
        if "minyak kedelai" in name:
            return "M_minyak_kedelai"
        if "zaitun" in name:
            return "M_minyak_zaitun"

    return category + "_" + name.strip()


milp_df["variety_key"] = milp_df.apply(make_variety_key, axis=1)

In [16]:
weekly_max_days = {
    # LH
    "LH_cumi": 3,
    "LH_daging_sapi": 3,
    "LH_ayam": 3,
    "LH_ikan_lele": 3,
    "LH_belut": 3,
    "LH_ikan_mas": 3,

    # LN
    "LN_tempe": 3,
    "LN_tahu": 3,
    "LN_kacang_kedelai": 3,
    "LN_kacang_merah": 3,
    "LN_kacang_hijau": 3,

    # MP
    "MP_makaroni": 3,
    "MP_bihun": 3,
    "MP_nasi": 5,
    "MP_roti": 3,
    "MP_biskuit": 2,
    "MP_kentang": 3,

    # Fruit
    "B_duku": 3,
    "B_pepaya": 3,
    "B_apel": 3,
    "B_pisang": 3,

    # Vegetable
    "S_daun_melinjo": 2,
    "S_nangka_muda": 2,
    "S_kangkung": 3,
    "S_bayam": 3,
    "S_rebung": 3,
    "S_komak": 3,

    # Oil/fat
    "M_minyak_kacang": 3,
    "M_minyak_kelapa": 3,
    "M_minyak_kedelai": 3,
    "M_minyak_zaitun": 3,
}

In [17]:
# ============================================================
# 6. Weekly variety constraints - optimized without y[d, i]
# ============================================================

z = {}

for key, max_days in weekly_max_days.items():

    key_indices = milp_df[
        milp_df["variety_key"] == key
    ].index.tolist()

    if len(key_indices) == 0:
        print(f"Warning: no candidate food found for variety key: {key}")
        continue

    for d in days:
        z[(d, key)] = pulp.LpVariable(
            f"z_day{d}_{key}",
            lowBound=0,
            upBound=1,
            cat="Binary"
        )

        # If any food in this group is selected, z[d, key] must be 1
        model += (
            pulp.lpSum(x[(d, i)] for i in key_indices)
            <= max_units * len(key_indices) * z[(d, key)],
            f"group_presence_upper_day{d}_{key}"
        )

    # Maximum number of days this food group may appear in 7 days
    model += (
        pulp.lpSum(z[(d, key)] for d in days) <= max_days,
        f"weekly_max_{key}"
    )

In [18]:
# ============================================================
# 7. Objective function - optimized
# ============================================================

energy_dev_pos = {}
energy_dev_neg = {}

for d in days:
    energy_dev_pos[d] = pulp.LpVariable(
        f"energy_dev_pos_day{d}",
        lowBound=0
    )

    energy_dev_neg[d] = pulp.LpVariable(
        f"energy_dev_neg_day{d}",
        lowBound=0
    )

    model += (
        daily_energy[d] - energy_target
        == energy_dev_pos[d] - energy_dev_neg[d],
        f"energy_deviation_day{d}"
    )

model += (
    pulp.lpSum(
        energy_dev_pos[d] + energy_dev_neg[d]
        for d in days
    )
    + 0.01 * pulp.lpSum(
        daily_sodium[d]
        for d in days
    )
    + 0.001 * pulp.lpSum(
        x[(d, i)]
        for d in days
        for i in milp_df.index
    )
), "minimize_energy_deviation_sodium_and_total_portion_units"

In [ ]:
# ============================================================
# 8. Solve model
# ============================================================

solver = pulp.PULP_CBC_CMD(msg=True, timeLimit=300)
model.solve(solver)

print("Solver status:", pulp.LpStatus[model.status])
print("Objective value:", pulp.value(model.objective))

In [ ]:
# ============================================================
# 9. Extract selected 7-day menu
# ============================================================

weekly_selected_items = []

for d in days:
    for i in milp_df.index:
        val = pulp.value(x[(d, i)])

        if val is not None and val > 0:
            row = milp_df.loc[i].copy()
            row["day"] = d
            row["selected_portions"] = val * portion_step
            row["selected_gram"] = row["gram_per_portion"] * row["selected_portions"]
            weekly_selected_items.append(row)

weekly_menu = pd.DataFrame(weekly_selected_items)

print("Number of selected food rows:", len(weekly_menu))

weekly_menu.to_csv("weekly_menu.csv", index=False)
display(
    weekly_menu[
        [
            "day",
            "food_name",
            "category_code",
            "variety_key",
            "urt",
            "gram_per_portion",
            "selected_portions",
            "selected_gram",
            "energy_kcal_per_portion",
            "protein_g_per_portion",
            "fat_g_per_portion",
            "carb_g_per_portion",
            "sodium_mg_per_portion",
            "fiber_g_per_portion"
        ]
    ].sort_values(["day", "category_code", "food_name"])
)


Number of selected food rows: 102


,day,food_name,category_code,variety_key,urt,gram_per_portion,selected_portions,selected_gram,energy_kcal_per_portion,protein_g_per_portion,fat_g_per_portion,carb_g_per_portion,sodium_mg_per_portion,fiber_g_per_portion
57,1,"Duku, segar",B,B_duku,14 buah sedang,80.0,2.0,160.0,50.40,0.800,0.160,12.880,1.60,3.440
10,1,"Cumi-cumi, segar",LH,LH_cumi,1 ekor kecil,45.0,1.5,67.5,33.75,7.245,0.315,0.045,16.65,0.000
14,1,"Sapi, daging, gemuk, segar",LH,LH_daging_sapi,1 potong sedang,35.0,1.5,52.5,95.55,6.125,7.700,0.000,32.55,0.000
22,1,"Kacang kedelai, segar",LN,LN_kacang_kedelai,2 Sendok Makan,25.0,1.0,25.0,71.50,7.550,3.900,7.525,7.00,0.725
21,1,"Tempe kedelai, mentah",LN,LN_tempe,2 potong sedang,50.0,2.0,100.0,100.50,10.400,4.400,6.750,4.50,0.700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3,7,Biskuit,MP,MP_biskuit,4 Buah Besar,40.0,1.0,40.0,183.20,2.760,5.760,30.040,8.12,0.840
26,7,"Daun kedondong, segar",S,"S_daun kedondong, segar",1 gelas,100.0,0.5,50.0,59.00,3.500,0.300,13.400,12.10,10.700
29,7,"Komak, segar",S,S_komak,1 gelas,100.0,0.5,50.0,99.00,3.300,0.100,21.200,15.00,10.300
38,7,"Labu siam, segar",S,"S_labu siam, segar",1 gelas,100.0,1.5,150.0,30.00,0.600,0.100,6.700,2.70,6.200


In [ ]:
# ============================================================
# 10. Daily nutrition validation
# ============================================================

daily_summary = []

for d in days:
    day_menu = weekly_menu[weekly_menu["day"] == d]

    total_energy = (day_menu["energy_kcal_per_portion"] * day_menu["selected_portions"]).sum()
    total_protein = (day_menu["protein_g_per_portion"] * day_menu["selected_portions"]).sum()
    total_fat = (day_menu["fat_g_per_portion"] * day_menu["selected_portions"]).sum()
    total_carb = (day_menu["carb_g_per_portion"] * day_menu["selected_portions"]).sum()
    total_sodium = (day_menu["sodium_mg_per_portion"] * day_menu["selected_portions"]).sum()
    total_fiber = (day_menu["fiber_g_per_portion"] * day_menu["selected_portions"]).sum()

    daily_summary.append({
        "day": d,
        "energy_kcal": total_energy,
        "protein_g": total_protein,
        "fat_g": total_fat,
        "carb_g": total_carb,
        "sodium_mg": total_sodium,
        "fiber_g": total_fiber,
        "energy_status": "pass" if energy_min <= total_energy <= energy_max else "fail",
        "carb_status": "pass" if total_carb >= constraints["carb_min_g"] else "fail",
        "sodium_status": "pass" if total_sodium <= constraints["sodium_max_mg"] else "fail",
        "fiber_status": "pass" if total_fiber >= constraints["fiber_min_g"] else "fail"
    })

daily_summary_df = pd.DataFrame(daily_summary)

display(daily_summary_df)

,day,energy_kcal,protein_g,fat_g,carb_g,sodium_mg,fiber_g,energy_status,carb_status,sodium_status,fiber_status
0,1,1600.050,69.3800,48.1825,235.5625,220.550,40.505,pass,pass,pass,pass
1,2,1600.000,81.3600,32.0300,253.1000,152.950,41.955,pass,pass,pass,pass
2,3,1600.150,64.6825,26.4700,279.7975,149.350,40.085,pass,pass,pass,pass
3,4,1600.000,71.9400,26.0100,273.5000,130.100,40.025,pass,pass,pass,pass
4,5,1600.000,78.5800,52.2800,214.8050,167.640,42.520,pass,pass,pass,pass
5,6,1599.975,72.5775,51.6575,218.5125,242.975,40.185,pass,pass,pass,pass
6,7,1600.275,59.6750,30.2575,280.4500,130.070,40.055,pass,pass,pass,pass


In [ ]:
# ============================================================
# 11. Daily category portion validation
# ============================================================

daily_category_check = (
    weekly_menu
    .groupby(["day", "category_code"])["selected_portions"]
    .sum()
    .reset_index(name="selected_portions_total")
)

required_rows = []

for d in days:
    for category, required_portion in adjusted_plan.items():
        required_rows.append({
            "day": d,
            "category_code": category,
            "required_portions": required_portion
        })

required_df = pd.DataFrame(required_rows)

daily_category_check = required_df.merge(
    daily_category_check,
    on=["day", "category_code"],
    how="left"
).fillna(0)

daily_category_check["status"] = np.where(
    daily_category_check["required_portions"] == daily_category_check["selected_portions_total"],
    "pass",
    "fail"
)

display(daily_category_check)

,day,category_code,required_portions,selected_portions_total,status
0,1,MP,4,4.0,pass
1,1,LH,3,3.0,pass
2,1,LN,3,3.0,pass
3,1,S,3,3.0,pass
4,1,B,2,2.0,pass
5,1,SS,0,0.0,pass
6,1,M,3,3.0,pass
7,1,G,0,0.0,pass
8,2,MP,4,4.0,pass
9,2,LH,3,3.0,pass
